In [ ]:
# Ô 0-1: gói (trl 0.29.1 - transformers 5.18.0 - peft 0.21.1, gỡ torchao — cùng bộ đã chạy lượt 299/304) + Java cho SPICE
import subprocess, sys, os, time, glob, json, shutil, re
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q 'transformers==5.18.0' 'peft==0.21.1' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap sacrebleu")
sh(f"{sys.executable} -m pip uninstall -y torchao")
sh(f'{sys.executable} -c "import torch, transformers, peft, trl; print(transformers.__version__, peft.__version__, trl.__version__, torch.cuda.device_count())"')
import torch
assert torch.cuda.device_count() == 2, "⛔ cần T4 x2 - đổi Accelerator rồi chạy lại"

In [ ]:
# Ô 0-2: đường dẫn, chép mã, hoà S1, hàm chạy nền
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"
def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g: return g[0]
    raise FileNotFoundError(duoi)

SRC = os.path.dirname(tim("ra_gen.py")); D312 = os.path.dirname(tim("ra_grpo_g0.py"))
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
shutil.copytree(SRC, RA, dirs_exist_ok=True)
for f in os.listdir(D312): shutil.copy(f"{D312}/{f}", RA)
print("SRC", SRC, "\nD312", D312, "\nBUNDLE", BUNDLE)
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", M], cwd=RA, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0
V = dict(recs=f"{BUNDLE}/p1_val_rows.jsonl", ocr=f"{BUNDLE}/ocr.jsonl", img=f"{BUNDLE}/images", ex=f"{SRC}/ex_val_k4.jsonl")
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def chay(script, args, log, gpu):
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}, cwd=RA, start_new_session=True)
def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120); print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h", [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]
def py(*a):
    r = subprocess.run(["python", *a], capture_output=True, text=True, cwd=RA); print(r.stdout[-5000:], r.stderr[-2000:]); assert r.returncode == 0

In [ ]:
# Ô G-1: sinh — greedy có khối (2 GPU), rồi 8 mẫu có khối (GPU0) song song không khối (GPU1) trên cùng 300 bước val
def lenh(mode, out, shard="0/1", khoi=True):
    a = ["--mode", mode, "--merged", M, "--recs", V["recs"], "--ocr", V["ocr"], "--images", V["img"], "--out", f"{W}/{out}", "--shard", shard]
    return a + (["--ra", V["ex"]] if khoi else [])

ps = [chay("ra_grpo_g0.py", lenh("greedy", f"g0_greedy_ra_{i}.jsonl", f"{i}/2"), f"{W}/g0_greedy_{i}.log", i) for i in (0, 1)]
doi(ps, [f"{W}/g0_greedy_{i}.log" for i in (0, 1)])
ps = [chay("ra_grpo_g0.py", lenh("mau", "g0_mau_ra.jsonl"), f"{W}/g0_mau_ra.log", 0),
      chay("ra_grpo_g0.py", lenh("mau", "g0_mau_khong.jsonl", khoi=False), f"{W}/g0_mau_khong.log", 1)]
doi(ps, [f"{W}/g0_mau_ra.log", f"{W}/g0_mau_khong.log"])
with open(f"{W}/g0_greedy_ra.jsonl", "w") as f:
    for i in (0, 1): f.write(open(f"{W}/g0_greedy_ra_{i}.jsonl").read())
print({p: sum(1 for _ in open(f"{W}/{p}")) for p in ("g0_greedy_ra.jsonl", "g0_mau_ra.jsonl", "g0_mau_khong.jsonl")})


In [ ]:
# Ô G-2: chấm greedy và đọc cổng
S1V = tim("pred_S1_vallon.jsonl")
py(f"{RA}/ra_score.py", "--recs", V["recs"], "--pred", f"S1={S1V}", "--pred", f"S1_ra={W}/g0_greedy_ra.jsonl",
   "--pred", f"ck500={SRC}/pred_ck500_vallon.jsonl", "--base", "S1", "--ra", f"S1_ra={V['ex']}", "--no-bert", "--out", f"{W}/diem_g0_greedy.json")
py(f"{RA}/ra_grpo_g0_doc.py", "--recs", V["recs"], "--ra", V["ex"], "--mau-ra", f"{W}/g0_mau_ra.jsonl",
   "--mau-khong", f"{W}/g0_mau_khong.jsonl", "--diem-greedy", f"{W}/diem_g0_greedy.json", "--out", f"{W}/g0_ket_qua.json")
shutil.rmtree(M, ignore_errors=True); shutil.rmtree(RA, ignore_errors=True)
print(f"{(time.time()-T0)/3600:.2f} h")

In [ ]:
# Ô G-3: nén kết quả
import shutil, os
shutil.make_archive("/tmp/ket_qua_312_G0", "zip", W, ".")
shutil.move("/tmp/ket_qua_312_G0.zip", f"{W}/ket_qua_312_G0.zip")
print(sorted(os.listdir(W)))
print(open(f"{W}/g0_ket_qua.json").read()[:3000])